In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"   # see issue #152
os.environ["CUDA_VISIBLE_DEVICES"]="4"
DEVICE = "cuda"

In [ ]:
from IPython.display import display
from tqdm import tqdm
import numpy as np

Notes: https://notebook.aidandevelops.com/Lab%20Notebook/2026%20-%20May/24_05_2026%209_27%20PM%20-%20Unsupervised%20dataset%20notes/

In [ ]:
from pathlib import Path
from pydantic import BaseModel, ValidationError
import yaml

In [ ]:
from open_vocab_mot import UNSUPERVISED_DATASET_INPUT_PATH, UNSUPERVISED_DATASET_OUTPUT_PATH

print(UNSUPERVISED_DATASET_INPUT_PATH)
assert UNSUPERVISED_DATASET_INPUT_PATH.exists()

print(UNSUPERVISED_DATASET_OUTPUT_PATH)
assert UNSUPERVISED_DATASET_OUTPUT_PATH.exists()

In [ ]:
from aidan_lib.video_utils.load_batched_frames import load_batched_frames, load_constrained_batched_frames
from aidan_lib.video_utils.scene_split import get_constrained_scenes, get_transnet_model
from aidan_lib.video_utils.video_data import get_video_data
from aidan_lib.models.sam3_video import generate_video_segmentation, SAM3Harness

In [ ]:
from typing import Iterator
class VideoDirConfig(BaseModel):
    prompts: list[str]

class VideoConfig(BaseModel):
    added_prompts: list[str]
    removed_prompts: list[str]

def combine_prompts(base_prompts: list[str], added_prompts: list[str], removed_prompts: list[str]):
    prompts = set(base_prompts)
    prompts.update(added_prompts)
    prompts.difference_update(removed_prompts)
    return list(prompts)

class VideoData(BaseModel):
    path: Path
    config: VideoConfig | None

class VideoDirData(BaseModel):
    path: Path
    config: VideoDirConfig
    videos: list[VideoData]

class VideoDirs(BaseModel):
    dirs_data: list[VideoDirData]

    def iter_videos(self) -> Iterator[tuple[VideoDirData, VideoData]]:
        for dir_data in self.dirs_data:
            for video_data in dir_data.videos:
                yield dir_data, video_data

ALLOWED_VIDEO_EXTENSIONS = [".mp4", ".webm"]

def ingest_dataset_paths(ds_root: Path, allowed_video_extensions: list[str]) -> VideoDirs:
    for i, video_ext in enumerate(allowed_video_extensions):
        if video_ext[0] != ".":
            print(f"WARNING: Allowed video extension {video_ext} does not include a . as the first character. This is invalid. Adding one.")
            allowed_video_extensions[i] = f".{video_ext}"

    video_dirs: list[VideoDirData] = []
    for video_dir in ds_root.iterdir():
        if not video_dir.is_dir():
            print(f"Skipping file {video_dir} as it is not a directory")
            continue

        config_file_path = video_dir / "config.yaml"
        if not config_file_path.exists():
            print(f"Skipping directory {video_dir} because it does not contain a config.yaml")
            continue

        with open(config_file_path, 'r') as f:
            config_dict = yaml.safe_load(f)

        try:
            video_dir_config = VideoDirConfig.model_validate(config_dict)
        except ValidationError as e:
            print(f"Failed to validate config for video dir {video_dir}")
            raise e

        video_datas: list[VideoData] = []
        for video_file in video_dir.iterdir():
            if not video_file.is_file():
                print(f"Skipping file {video_file} as it is not a file")
                continue

            if video_file.suffix not in allowed_video_extensions:
                print(f"Skipping file {video_file} as its file extension {video_file.suffix} is not an allowed extension {allowed_video_extensions}")
                continue

            config_file = video_file.parent / f"{video_file.stem}.yaml"
            if config_file.exists():
                with open(config_file, 'r') as f:
                    config_dict = yaml.safe_load(f)

                try:
                    video_config = VideoConfig.model_validate(config_dict)
                except ValidationError as e:
                    print(f"Failed to validate config for video {video_file}")
                    raise e
            else:
                video_config = None

            video_data = VideoData(path=video_file, config=video_config)
            video_datas.append(video_data)
        
        video_dir_data = VideoDirData(
            path=video_dir,
            config=video_dir_config,
            videos=video_datas
        )
        video_dirs.append(video_dir_data)
    
    return VideoDirs(dirs_data=video_dirs)

In [ ]:
test_video_dir = UNSUPERVISED_DATASET_INPUT_PATH / "tests"
test_video_dir.mkdir(exist_ok=True)

test_video_path = test_video_dir / "all_creatures_great_an_small.webm"
if not test_video_path.exists():
    test_video_url = "https://www.youtube.com/watch?v=cPtk-gXedAs"
    test_video_path_str = str(test_video_path)

    !uvx --no-cache yt-dlp -o {test_video_path_str} {test_video_url} 

video_cfg_path = test_video_path.parent / f"{test_video_path.stem}.yaml"
if not video_cfg_path.exists():
    with open(video_cfg_path, "w") as f:
        f.writelines([
            "added_prompts:\n",
            "  - Cow\n",
            "  - Chicken\n",
            "\n",
            "removed_prompts:\n",
            "  - Dog\n"
        ])

cfg_path = test_video_dir / "config.yaml"
if not cfg_path.exists():
    with open(cfg_path, "w") as f:
        f.writelines([
            "prompts:\n",
            "  - Person\n",
            "  - Dog\n"
        ])

In [ ]:
video_dirs = ingest_dataset_paths(UNSUPERVISED_DATASET_INPUT_PATH, ALLOWED_VIDEO_EXTENSIONS)

In [ ]:
print(video_dirs.model_dump_json(indent=2))

In [ ]:
for dir_data, video_data in video_dirs.iter_videos():
    print(dir_data.config)
    print(video_data.config)
    print(video_data.path)
    prompts = combine_prompts(
        dir_data.config.prompts,
        video_data.config.added_prompts,
        video_data.config.removed_prompts
    )
    print(prompts)

In [ ]:
sam = SAM3Harness(max_num_objects=64)

In [ ]:
transnet = get_transnet_model("cuda")
constrained_scenes = get_constrained_scenes(test_video_path, transnet, threshold=0.75)

In [ ]:
batch_frame_loader = load_constrained_batched_frames(test_video_path, constrained_scenes, batch_size=120, skip_frames=5, convert_pil=True, overlap=1)

In [ ]:
from aidan_lib.visualization.segmentations import visualize_segmentations, int_mask_to_binary_masks

In [ ]:
prompts = ["Person", "Light"]
frame_seg_generator = generate_video_segmentation(
    harness=sam,
    prompts=prompts,
    batch_frame_loader=batch_frame_loader
)

In [ ]:
fps = 15
from tqdm import tqdm
import imageio

# Use imageio"s writer in a context manager to ensure it closes properly
output_path = Path("./test.mp4")
progress = tqdm()
with imageio.get_writer(output_path, fps=fps, format="mp4", codec="libx264") as writer:
    for frame_info in frame_seg_generator:
        # Unpack the FrameSegmentationInfo
        frame_num, frame, sam_seg, background_index, obj_id_to_prompt = frame_info
        
        # Convert segmentation to binary masks
        masks, obj_ids = int_mask_to_binary_masks(sam_seg, background_index=background_index)
        obj_prompts = [obj_id_to_prompt.get(obj_id, "UNKNOWN") for obj_id in obj_ids]
        
        # Generate labels dynamically for however many objects are in the frame
        labels = [f"{obj_prompt} {obj_id}" for obj_prompt, obj_id in zip(obj_prompts, obj_ids)]
        
        # Create the visualization (img is a PIL Image)
        img = visualize_segmentations(frame, masks, labels=labels)
        
        # Convert the PIL Image to a NumPy array for imageio
        frame_array = np.expand_dims(np.array(img), axis=0)
        
        # Write the frame to the video file
        writer.append_data(frame_array)
        
        progress.update(1)
        progress.set_description(f"Frame {frame_num}")

print(f"Video saved successfully to {output_path.absolute()}")

In [ ]:
transnet = get_transnet_model("cuda")

In [ ]:
from shutil import rmtree
import imageio
from aidan_lib.visualization.segmentations import visualize_segmentations, int_mask_to_binary_masks
from PIL import Image
import yaml

batch_size = 120
skip_frames = 5
overlap = 1
create_demo_videos = True
edge_width = 15
verbose = True
for dir_data, video_data in video_dirs.iter_videos():
    video_path = video_data.path
    video_parent_relpath = str(video_path.parent.relative_to(UNSUPERVISED_DATASET_INPUT_PATH))
    base_out_dir = UNSUPERVISED_DATASET_OUTPUT_PATH / video_parent_relpath / video_path.stem
    base_out_dir.mkdir(exist_ok=True, parents=True)

    video_info = get_video_data(video_path)

    prompts = combine_prompts(
        dir_data.config.prompts,
        video_data.config.added_prompts,
        video_data.config.removed_prompts
    )

    # We keep intermediate results in temp files so that we can tell if it actually finished
    demo_video_path = base_out_dir / f"{video_path.stem}_segs.mp4"
    tmp_demo_video_path = base_out_dir / f"{video_path.stem}_segs_tmp.mp4"
    has_demo_video = demo_video_path.exists()
    needs_demo_video = not has_demo_video and create_demo_videos
    if tmp_demo_video_path.exists():
        print(f"Removing old temp demo video")
        tmp_demo_video_path.unlink(missing_ok=True)

    full_segmentations_dir = base_out_dir / "full_frame_segmentations"
    tmp_full_segmentations_dir = base_out_dir / "full_frame_segmentations_tmp"
    has_full_segmentations = full_segmentations_dir.exists()
    if tmp_full_segmentations_dir.exists():
        print(f"Removing old temp full frame segmentations dir")
        rmtree(tmp_full_segmentations_dir, ignore_errors=True)
    tmp_full_segmentations_dir.mkdir()

    cropped_segmentations_dir = base_out_dir / "cropped_segmentations"
    tmp_cropped_segmentations_dir = base_out_dir / "cropped_segmentations_tmp"
    has_cropped_segmentations = cropped_segmentations_dir.exists()
    if tmp_cropped_segmentations_dir.exists():
        print(f"Removing old temp cropped segmentations dir")
        rmtree(tmp_cropped_segmentations_dir, ignore_errors=True)
    tmp_cropped_segmentations_dir.mkdir()

    needs_processing = not (has_demo_video and has_full_segmentations and has_cropped_segmentations)

    if not needs_processing:
        print(f"Video {video_path} already has results in {base_out_dir}")
        continue

    print(f"Processing video {video_path} to {base_out_dir} with prompts {prompts}")

    constrained_scenes = get_constrained_scenes(video_path, transnet, threshold=0.75)

    batch_frame_loader = load_constrained_batched_frames(
        video_path,
        constrained_scenes,
        batch_size=batch_size,
        skip_frames=skip_frames,
        convert_pil=True,
        overlap=overlap
    )

    frame_seg_generator = generate_video_segmentation(
        harness=sam,
        prompts=prompts,
        batch_frame_loader=batch_frame_loader
    )

    try:
        frames_to_process = video_info.frame_count // skip_frames
        progress = tqdm(total=frames_to_process, disable=not verbose)
        if needs_demo_video:
            demo_video_frame_rate = video_info.fps / skip_frames if skip_frames is not None else video_info.fps
            demo_video_writer = imageio.get_writer(tmp_demo_video_path, fps=demo_video_frame_rate, format="mp4", codec="libx264")
        else:
            demo_video_writer = None

        visible_seg_map: dict[int, list[int]] = {}  # Which tracklets are visible on each frame
        known_negatives: set[tuple[int, int]] = set()
        global_id_to_prompt: dict[int, str] = {}
        for frame_info in frame_seg_generator:
            frame_num, frame, sam_seg, background_index, obj_id_to_prompt = frame_info
            progress.update(1)
            progress.set_description(f"Frame {frame_num}/{video_info.frame_count}")

            # We have a utility that gives us the unique ids with their corresponding maps
            masks, obj_ids = int_mask_to_binary_masks(sam_seg, background_index=background_index)
            obj_prompts = [obj_id_to_prompt.get(obj_id, "UNKNOWN") for obj_id in obj_ids]
            
            # We can use the unique ids to populate the visible seg map for this frame
            visible_seg_map[frame_num] = obj_ids

            # Update the global map from id to prompt
            for obj_id in obj_ids:
                global_id_to_prompt[obj_id] = obj_id_to_prompt.get(obj_id, "UNKNOWN")

            # Using the constraint that two objects cannot be the same if they appear at the same time
            # we can populate some known negatives
            # We use the ordering (lower, higher) to avoid duplicate negatives due to the undirected graph of negatives
            for i in range(len(obj_ids)):
                for j in range(i+1, len(obj_ids)):
                    lower = min(obj_ids[i], obj_ids[j])
                    higher = max(obj_ids[i], obj_ids[j])
                    known_negatives.add((lower, higher))

            if needs_demo_video:
                # Then we need to visualize the segmentations and save them
                labels = [f"{obj_prompt} {obj_id}" for obj_prompt, obj_id in zip(obj_prompts, obj_ids)]
                segmented_frame = visualize_segmentations(frame, masks, labels=labels)
                segmented_frame_array = np.expand_dims(np.array(segmented_frame), axis=0)
                demo_video_writer.append_data(segmented_frame_array)

            if not has_full_segmentations:
                # Then we need to save the full frames and segmentations
                frame_path = tmp_full_segmentations_dir / f"frame_{frame_num}.jpg"
                frame.save(frame_path)

                # # For saving in the numpy ecosystem
                # segmentation_path = tmp_full_segmentations_dir / f"frame_{frame_num}_seg.npz"
                # np.savez_compressed(segmentation_path, sam_seg)

                # For saving as an image
                segmentation_path = tmp_full_segmentations_dir / f"frame_{frame_num}_seg.png"
                seg_img = Image.fromarray(sam_seg)
                seg_img.save(segmentation_path)

            if not has_cropped_segmentations:
                frame_w, frame_h = frame.size

                # Then we will save cropped versions of each of the tracklets
                for mask, obj_id in zip(masks, obj_ids):
                    # Find the rows and columns where the mask is active
                    rows = np.any(mask, axis=1)
                    cols = np.any(mask, axis=0)
                    
                    # Prevent errors if a mask is completely empty in this frame
                    if not np.any(rows) or not np.any(cols):
                        continue
                        
                    # Get the initial bounding box indices
                    rmin, rmax = np.where(rows)[0][[0, -1]]
                    cmin, cmax = np.where(cols)[0][[0, -1]]
                    
                    # Apply edge_width padding while keeping within frame boundaries
                    rmin = max(0, rmin - edge_width)
                    rmax = min(frame_h, rmax + edge_width + 1)
                    cmin = max(0, cmin - edge_width)
                    cmax = min(frame_w, cmax + edge_width + 1)
                    
                    # Crop the original frame using PIL (left, upper, right, lower)
                    cropped_frame = frame.crop((cmin, rmin, cmax, rmax))
                    
                    # Crop the mask array and convert it to a PIL image
                    cropped_mask_array = mask[rmin:rmax, cmin:cmax]
                    # Convert boolean or 0/1 mask to 0-255 grayscale for saving
                    cropped_mask_img = Image.fromarray((cropped_mask_array * 255).astype(np.uint8))
                    
                    # Ensure the output directory for this specific ID exists
                    obj_dir = tmp_cropped_segmentations_dir / f"id_{obj_id}"
                    obj_dir.mkdir(parents=True, exist_ok=True)
                    
                    # Save the cropped frame and cropped mask
                    cropped_frame.save(obj_dir / f"frame_{frame_num}.jpg")
                    cropped_mask_img.save(obj_dir / f"frame_{frame_num}_mask.png")

        
        # --- GENERATE METADATA ---
        # 1. Map mutually exclusive negatives per ID
        negatives_map: dict[int, list[int]] = {obj_id: [] for obj_id in global_id_to_prompt.keys()}
        for id1, id2 in known_negatives:
            negatives_map[id1].append(id2)
            negatives_map[id2].append(id1)
            
        # 2. Save cropped segmentations metadata
        if not has_cropped_segmentations:
            for obj_id, prompt in global_id_to_prompt.items():
                obj_dir = tmp_cropped_segmentations_dir / f"id_{obj_id}"
                if obj_dir.exists():
                    meta = {
                        "prompt": prompt,
                        "negatives": negatives_map.get(obj_id, [])
                    }
                    with open(obj_dir / "metadata.yaml", "w") as f:
                        yaml.dump(meta, f, default_flow_style=False)

        # 3. Save full frame metadata
        if not has_full_segmentations:
            full_meta = {
                "visible_seg_map": visible_seg_map,
                "known_negatives": [list(pair) for pair in known_negatives], # YAML prefers lists over tuples
                "id_to_prompt": global_id_to_prompt
            }
            with open(tmp_full_segmentations_dir / "metadata.yaml", "w") as f:
                yaml.dump(full_meta, f, default_flow_style=False)
        # -------------------------

        # Move the temp to the true paths
        if demo_video_writer is not None:
            tmp_demo_video_path.rename(demo_video_path)

        if not has_full_segmentations:
            tmp_full_segmentations_dir.rename(full_segmentations_dir)
            
        if not has_cropped_segmentations:
            tmp_cropped_segmentations_dir.rename(cropped_segmentations_dir)

        print(f"Finished processing video {video_path} to {base_out_dir}")
    finally:
        if demo_video_writer is not None:
            demo_video_writer.close()
            del demo_video_writer
